# MNIST 手写数字识别 —— 深度学习入门项目

本项目从零开始，使用 **PyTorch** 框架实现经典的 MNIST 手写数字识别任务。

**项目内容：**
1. 数据加载与可视化
2. 构建卷积神经网络（CNN）模型
3. 模型训练与评估
4. 结果可视化与模型保存

> MNIST 数据集包含 60,000 张训练图片和 10,000 张测试图片，每张图片是 28×28 像素的灰度手写数字（0-9）。

## 第一步：导入所需库

导入项目所需的所有 Python 库：
- **torch**：PyTorch 深度学习框架核心库
- **torch.nn**：神经网络模块，用于构建网络层
- **torch.optim**：优化器模块，用于更新模型参数
- **torchvision**：提供常用数据集和图像变换工具
- **matplotlib**：数据可视化绑图库
- **numpy**：科学计算库

In [ ]:
import torch                          # PyTorch 深度学习框架
import torch.nn as nn                 # 神经网络模块（定义层、损失函数等）
import torch.optim as optim           # 优化器模块（SGD、Adam 等）
import torch.nn.functional as F       # 函数式接口（激活函数、池化等）
from torch.utils.data import DataLoader  # 数据加载器，按批次加载数据
from torchvision import datasets, transforms  # 数据集和图像变换工具
import matplotlib.pyplot as plt       # 绑图库，用于数据可视化
import numpy as np                    # 科学计算库，处理数组和矩阵
from sklearn.metrics import confusion_matrix  # 混淆矩阵计算工具
import random                         # 随机数生成模块

# 设置 matplotlib 支持中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False   # 解决负号显示问题

print("所有库导入成功！")
print(f"PyTorch 版本: {torch.__version__}")

## 第二步：设置超参数与设备配置

定义训练过程中的关键超参数：
- **batch_size**：每个批次的样本数量，影响训练速度和内存占用
- **num_epochs**：训练轮数，即遍历整个训练集的次数
- **learning_rate**：学习率，控制参数更新的步长
- **device**：自动检测 GPU 是否可用，有 GPU 用 GPU，没有就用 CPU

In [ ]:
# ===== 超参数设置 =====
batch_size = 64         # 每个批次加载 64 张图片
num_epochs = 10         # 总共训练 10 个 epoch（轮次）
learning_rate = 0.001   # 学习率设为 0.001（Adam 优化器的常用默认值）
num_classes = 10        # MNIST 有 10 个类别（数字 0-9）

# ===== 设备配置 =====
# 检测是否有可用的 GPU（NVIDIA CUDA），有则用 GPU 加速，否则用 CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 打印超参数和设备信息
print(f"训练设备: {device}")
print(f"批量大小: {batch_size}")
print(f"训练轮数: {num_epochs}")
print(f"学习率: {learning_rate}")
print(f"分类数量: {num_classes}")

## 第三步：下载并加载 MNIST 数据集

使用 `torchvision.datasets.MNIST` 自动下载 MNIST 数据集：
- 训练集：60,000 张手写数字图片
- 测试集：10,000 张手写数字图片
- 每张图片为 28×28 像素的灰度图像
- 使用 `transforms.ToTensor()` 将图像转为 PyTorch 张量，并自动归一化到 [0, 1] 范围

In [ ]:
# 定义数据预处理/变换操作
# transforms.ToTensor()：将 PIL 图像或 numpy 数组转换为 FloatTensor，像素值从 [0,255] 归一化到 [0,1]
# transforms.Normalize((0.1307,), (0.3081,))：使用 MNIST 数据集的全局均值和标准差进行标准化
transform = transforms.Compose([
    transforms.ToTensor(),                        # 转为张量并归一化到 [0,1]
    transforms.Normalize((0.1307,), (0.3081,))    # 用均值0.1307和标准差0.3081标准化
])

# 下载并加载训练集
# root='./data'：数据保存路径
# train=True：加载训练集
# download=True：如果本地没有数据则自动下载
# transform=transform：应用上面定义的变换
train_dataset = datasets.MNIST(
    root='./data',          # 数据存储目录
    train=True,             # 加载训练集
    download=True,          # 自动下载
    transform=transform     # 应用数据变换
)

# 下载并加载测试集
test_dataset = datasets.MNIST(
    root='./data',          # 数据存储目录
    train=False,            # 加载测试集
    download=True,          # 自动下载
    transform=transform     # 应用数据变换
)

# 打印数据集基本信息
print(f"训练集样本数量: {len(train_dataset)}")
print(f"测试集样本数量: {len(test_dataset)}")
print(f"图像尺寸: {train_dataset[0][0].shape}")   # 打印第一张图的张量形状
print(f"标签示例: {train_dataset[0][1]}")           # 打印第一张图的标签

## 第四步：数据可视化与探索

在训练模型之前，先直观地看一看数据长什么样：
1. 随机展示 3×3 共 9 张手写数字图片，感受数据集的内容
2. 统计 0-9 各类别的样本数量，查看数据分布是否均衡

In [ ]:
# ===== 可视化部分1：随机展示 9 张手写数字图片 =====
fig, axes = plt.subplots(3, 3, figsize=(8, 8))  # 创建 3x3 的子图网格
fig.suptitle('MNIST 训练集样本展示', fontsize=16)  # 设置总标题

for i, ax in enumerate(axes.flatten()):          # 遍历每个子图
    idx = random.randint(0, len(train_dataset) - 1)  # 随机选择一个样本索引
    image, label = train_dataset[idx]            # 获取图像和标签
    # image 形状为 [1, 28, 28]，squeeze 去掉通道维度变成 [28, 28]
    ax.imshow(image.squeeze(), cmap='gray')      # 以灰度图方式显示
    ax.set_title(f'标签: {label}', fontsize=12)  # 在图片上方显示标签
    ax.axis('off')                               # 关闭坐标轴

plt.tight_layout()   # 自动调整子图间距
plt.show()           # 显示图片

In [ ]:
# ===== 可视化部分2：统计各类别样本数量分布 =====
# 获取训练集中所有样本的标签
train_labels = train_dataset.targets.numpy()  # 将标签张量转为 numpy 数组

# 统计每个数字（0-9）出现的次数
unique, counts = np.unique(train_labels, return_counts=True)  # 统计唯一值及其出现次数

# 绘制柱状图
plt.figure(figsize=(10, 5))                          # 设置画布大小
plt.bar(unique, counts, color='steelblue', edgecolor='black')  # 绘制柱状图
plt.xlabel('数字类别', fontsize=14)                   # x 轴标签
plt.ylabel('样本数量', fontsize=14)                   # y 轴标签
plt.title('MNIST 训练集各类别样本数量分布', fontsize=16)  # 标题
plt.xticks(unique)                                    # 设置 x 轴刻度为 0-9
# 在每个柱子顶部标注具体数量
for u, c in zip(unique, counts):
    plt.text(u, c + 50, str(c), ha='center', fontsize=10)  # 在柱顶显示数值
plt.tight_layout()
plt.show()

## 第五步：数据预处理与 DataLoader 创建

使用 `DataLoader` 将数据集包装为可按批次迭代的加载器：
- **训练集**：设置 `shuffle=True`，每个 epoch 打乱数据顺序，避免模型记住数据的排列顺序
- **测试集**：设置 `shuffle=False`，保持数据顺序不变，方便评估
- **batch_size**：每次取出 64 张图片组成一个批次

In [ ]:
# 创建训练集的 DataLoader
train_loader = DataLoader(
    dataset=train_dataset,   # 传入训练数据集
    batch_size=batch_size,   # 每个批次 64 个样本
    shuffle=True,            # 每个 epoch 开始时打乱数据
    num_workers=0            # 数据加载的子进程数（Windows 下建议设为 0）
)

# 创建测试集的 DataLoader
test_loader = DataLoader(
    dataset=test_dataset,    # 传入测试数据集
    batch_size=batch_size,   # 每个批次 64 个样本
    shuffle=False,           # 测试集不需要打乱
    num_workers=0            # 数据加载的子进程数
)

# 验证 DataLoader：取出一个批次查看数据形状
images, labels = next(iter(train_loader))  # 获取第一个批次的数据
print(f"一个批次的图像张量形状: {images.shape}")   # 期望输出: [64, 1, 28, 28]
print(f"一个批次的标签张量形状: {labels.shape}")   # 期望输出: [64]
print(f"图像像素值范围: [{images.min():.4f}, {images.max():.4f}]")  # 查看归一化后的值域

## 第六步：定义卷积神经网络（CNN）模型

构建一个经典的 CNN 网络用于图像分类：

**网络结构：**
| 层          | 说明                                |
|-------------|-------------------------------------|
| Conv2d(1→32) | 第一个卷积层，提取低级特征（边缘、纹理）|
| ReLU + MaxPool | 激活 + 下采样                      |
| Conv2d(32→64) | 第二个卷积层，提取高级特征           |
| ReLU + MaxPool | 激活 + 下采样                      |
| Flatten      | 将二维特征图展平为一维向量           |
| Linear(3136→128) | 第一个全连接层                  |
| ReLU         | 激活函数                            |
| Linear(128→10) | 输出层，10 个类别                  |

In [ ]:
class CNN(nn.Module):
    """卷积神经网络模型，用于 MNIST 手写数字分类"""

    def __init__(self):
        super(CNN, self).__init__()  # 调用父类构造函数

        # 第一个卷积层：输入通道=1（灰度图），输出通道=32，卷积核大小=3x3，padding=1保持尺寸不变
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        # 第二个卷积层：输入通道=32，输出通道=64，卷积核大小=3x3，padding=1
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        # 最大池化层：窗口大小=2x2，步长=2，将特征图尺寸缩小一半
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # 第一个全连接层：输入大小=64*7*7=3136（经过两次池化后的特征图展平），输出=128
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        # 第二个全连接层（输出层）：输入=128，输出=10（0-9十个数字类别）
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        """前向传播函数，定义数据在网络中的流动路径"""
        # x 输入形状: [batch_size, 1, 28, 28]

        # 第一个卷积块：卷积 → ReLU激活 → 最大池化
        x = self.pool(F.relu(self.conv1(x)))   # 输出形状: [batch_size, 32, 14, 14]

        # 第二个卷积块：卷积 → ReLU激活 → 最大池化
        x = self.pool(F.relu(self.conv2(x)))   # 输出形状: [batch_size, 64, 7, 7]

        # 展平操作：将多维特征图展平为一维向量
        x = x.view(x.size(0), -1)             # 输出形状: [batch_size, 3136]

        # 第一个全连接层 + ReLU 激活
        x = F.relu(self.fc1(x))                # 输出形状: [batch_size, 128]

        # 输出层（不需要 softmax，因为 CrossEntropyLoss 内部会处理）
        x = self.fc2(x)                        # 输出形状: [batch_size, 10]

        return x

# 实例化模型并移动到指定设备（GPU 或 CPU）
model = CNN().to(device)

# 打印模型结构
print(model)
print(f"\n{'='*50}")

# 计算并打印模型的总参数量
total_params = sum(p.numel() for p in model.parameters())           # 所有参数数量
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)  # 可训练参数数量
print(f"模型总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")

## 第七步：定义损失函数与优化器

- **损失函数**：使用 `CrossEntropyLoss`（交叉熵损失），它是多分类任务的标准损失函数，内部集成了 Softmax 操作
- **优化器**：使用 `Adam` 优化器，它结合了动量（Momentum）和自适应学习率（RMSProp）的优点，收敛速度快且稳定

In [ ]:
# 定义损失函数：交叉熵损失
# CrossEntropyLoss 将 Softmax 和负对数似然损失（NLLLoss）合并在一起
# 适用于多分类任务，输入为模型输出的 logits（未经 softmax 的原始值）
criterion = nn.CrossEntropyLoss()

# 定义优化器：Adam 优化器
# Adam 是目前最流行的优化器之一，自动调整每个参数的学习率
# model.parameters()：传入模型的所有可训练参数
# lr=learning_rate：设置初始学习率
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"损失函数: {criterion}")
print(f"优化器: {optimizer}")

## 第八步：模型训练

开始训练模型！训练流程如下：
1. **外层循环**：遍历每个 epoch（训练轮次）
2. **内层循环**：遍历每个 batch（批次）
3. **每个 batch 的操作**：
   - 将数据移到 GPU/CPU
   - 前向传播：计算模型输出
   - 计算损失
   - 反向传播：计算梯度
   - 更新参数
   - 清零梯度

训练过程中会记录每个 epoch 的损失和准确率。

In [ ]:
# 用于记录每个 epoch 的损失和准确率
train_losses = []      # 存储每个 epoch 的平均损失
train_accuracies = []  # 存储每个 epoch 的训练准确率

# 将模型设为训练模式（启用 Dropout、BatchNorm 等训练时行为）
model.train()

print("开始训练...")
print(f"{'='*60}")

# 外层循环：遍历每个 epoch
for epoch in range(num_epochs):
    running_loss = 0.0    # 累计当前 epoch 的总损失
    correct = 0           # 累计当前 epoch 的正确预测数
    total = 0             # 累计当前 epoch 的总样本数

    # 内层循环：遍历每个 batch
    for batch_idx, (images, labels) in enumerate(train_loader):
        # 将数据移到指定设备（GPU 或 CPU）
        images = images.to(device)    # 图像数据移到设备
        labels = labels.to(device)    # 标签数据移到设备

        # ===== 前向传播 =====
        outputs = model(images)        # 将图像输入模型，得到预测输出
        loss = criterion(outputs, labels)  # 计算预测值与真实标签之间的损失

        # ===== 反向传播与参数更新 =====
        optimizer.zero_grad()          # 清零之前累积的梯度（PyTorch 会累积梯度）
        loss.backward()                # 反向传播，计算每个参数的梯度
        optimizer.step()               # 根据梯度更新模型参数

        # ===== 统计训练指标 =====
        running_loss += loss.item()    # 累加当前 batch 的损失值
        _, predicted = torch.max(outputs.data, 1)  # 取预测概率最大的类别作为预测结果
        total += labels.size(0)        # 累加当前 batch 的样本数
        correct += (predicted == labels).sum().item()  # 统计预测正确的数量

        # 每 200 个 batch 打印一次训练进度
        if (batch_idx + 1) % 200 == 0:
            print(f'  Epoch [{epoch+1}/{num_epochs}], '
                  f'Batch [{batch_idx+1}/{len(train_loader)}], '
                  f'损失: {loss.item():.4f}')

    # 计算当前 epoch 的平均损失和准确率
    epoch_loss = running_loss / len(train_loader)     # 平均损失 = 总损失 / batch 数量
    epoch_acc = 100.0 * correct / total               # 准确率 = 正确数 / 总数 * 100%

    # 记录到列表中，用于后续绘图
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)

    # 打印当前 epoch 的训练结果
    print(f'Epoch [{epoch+1}/{num_epochs}] 完成 | '
          f'平均损失: {epoch_loss:.4f} | '
          f'训练准确率: {epoch_acc:.2f}%')
    print(f'{"-"*60}')

print(f"\n训练完成！最终训练准确率: {train_accuracies[-1]:.2f}%")

## 第九步：绘制训练损失与准确率曲线

通过可视化训练过程中的损失和准确率变化，可以直观地观察：
- **损失曲线**：是否在持续下降？是否收敛？
- **准确率曲线**：是否在持续上升？最终达到了多少？

In [ ]:
# 创建包含两个子图的画布
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ===== 子图1：训练损失曲线 =====
ax1.plot(range(1, num_epochs + 1), train_losses, 'b-o', linewidth=2, markersize=6)  # 蓝色实线+圆点
ax1.set_xlabel('Epoch（训练轮次）', fontsize=12)    # x 轴标签
ax1.set_ylabel('平均损失', fontsize=12)              # y 轴标签
ax1.set_title('训练损失变化曲线', fontsize=14)       # 标题
ax1.grid(True, alpha=0.3)                            # 显示网格线

# ===== 子图2：训练准确率曲线 =====
ax2.plot(range(1, num_epochs + 1), train_accuracies, 'r-o', linewidth=2, markersize=6)  # 红色实线+圆点
ax2.set_xlabel('Epoch（训练轮次）', fontsize=12)    # x 轴标签
ax2.set_ylabel('准确率 (%)', fontsize=12)            # y 轴标签
ax2.set_title('训练准确率变化曲线', fontsize=14)     # 标题
ax2.grid(True, alpha=0.3)                            # 显示网格线

plt.tight_layout()  # 自动调整布局
plt.show()

## 第十步：模型测试与评估

在测试集上评估训练好的模型：
- 使用 `model.eval()` 切换到评估模式（关闭 Dropout 等）
- 使用 `torch.no_grad()` 关闭梯度计算，节省内存和计算时间
- 计算整体测试准确率和每个类别的分类准确率

In [ ]:
# 将模型切换到评估模式
model.eval()

# 初始化统计变量
test_correct = 0        # 测试集上的正确预测数
test_total = 0          # 测试集上的总样本数
test_loss = 0.0         # 测试集上的总损失

# 每个类别的统计变量
class_correct = [0] * num_classes   # 每个类别的正确预测数
class_total = [0] * num_classes     # 每个类别的总样本数

# 收集所有的真实标签和预测标签（用于后续混淆矩阵）
all_labels = []        # 所有真实标签
all_predictions = []   # 所有预测标签

# 关闭梯度计算，评估时不需要计算梯度
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)    # 将图像移到设备
        labels = labels.to(device)    # 将标签移到设备

        outputs = model(images)        # 前向传播，得到预测结果
        loss = criterion(outputs, labels)  # 计算损失

        test_loss += loss.item()       # 累加损失
        _, predicted = torch.max(outputs, 1)  # 取概率最大的类别为预测结果

        test_total += labels.size(0)   # 累加总样本数
        test_correct += (predicted == labels).sum().item()  # 累加正确预测数

        # 收集标签和预测结果
        all_labels.extend(labels.cpu().numpy())          # 转为 numpy 并收集
        all_predictions.extend(predicted.cpu().numpy())  # 转为 numpy 并收集

        # 统计每个类别的正确率
        for i in range(labels.size(0)):
            label = labels[i].item()           # 获取当前样本的真实标签
            class_total[label] += 1            # 该类别总数+1
            if predicted[i] == labels[i]:      # 如果预测正确
                class_correct[label] += 1      # 该类别正确数+1

# 计算并打印整体测试结果
test_accuracy = 100.0 * test_correct / test_total
avg_test_loss = test_loss / len(test_loader)
print(f"测试集上的总体结果:")
print(f"  平均损失: {avg_test_loss:.4f}")
print(f"  准确率: {test_accuracy:.2f}% ({test_correct}/{test_total})")
print(f"\n{'='*40}")
print(f"各类别（数字 0-9）的分类准确率:")
print(f"{'='*40}")

# 打印每个数字类别的准确率
for i in range(num_classes):
    if class_total[i] > 0:
        acc = 100.0 * class_correct[i] / class_total[i]
        print(f"  数字 {i}: {acc:.2f}% ({class_correct[i]}/{class_total[i]})")

## 第十一步：混淆矩阵可视化

混淆矩阵可以清楚地展示模型将哪些数字搞混了：
- 对角线上的值表示正确预测的数量
- 非对角线上的值表示被错误分类的数量
- 颜色越深表示数量越多

In [ ]:
# 使用 sklearn 计算混淆矩阵
cm = confusion_matrix(all_labels, all_predictions)  # 传入真实标签和预测标签

# 绘制混淆矩阵热力图
plt.figure(figsize=(10, 8))                                  # 设置画布大小
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)   # 用蓝色色彩映射绘制热力图
plt.title('混淆矩阵（Confusion Matrix）', fontsize=16)       # 标题
plt.colorbar()                                                # 显示颜色条

# 设置坐标轴刻度
tick_marks = np.arange(num_classes)         # 生成 0-9 的刻度位置
plt.xticks(tick_marks, tick_marks)          # 设置 x 轴刻度标签
plt.yticks(tick_marks, tick_marks)          # 设置 y 轴刻度标签

# 在每个格子中标注数值
thresh = cm.max() / 2.0                    # 颜色阈值，用于决定文字颜色（深底白字/浅底黑字）
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, format(cm[i, j], 'd'),              # 在 (j, i) 位置显示数值
                 horizontalalignment="center",               # 水平居中
                 color="white" if cm[i, j] > thresh else "black")  # 深背景用白字

plt.ylabel('真实标签', fontsize=14)       # y 轴标签
plt.xlabel('预测标签', fontsize=14)       # x 轴标签
plt.tight_layout()
plt.show()

## 第十二步：单张图片预测与展示

从测试集中随机选取若干张图片，使用训练好的模型进行预测：
- **绿色标题**：模型预测正确
- **红色标题**：模型预测错误

通过这种方式可以直观地感受模型的识别效果。

In [ ]:
# 确保模型处于评估模式
model.eval()

# 创建 4x4 的子图网格，展示 16 张预测结果
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('模型预测结果展示', fontsize=18)

for i, ax in enumerate(axes.flatten()):
    # 随机选择一个测试样本
    idx = random.randint(0, len(test_dataset) - 1)
    image, true_label = test_dataset[idx]        # 获取图像和真实标签

    # 用模型进行预测
    with torch.no_grad():                         # 不计算梯度
        # image.unsqueeze(0)：增加 batch 维度，从 [1,28,28] 变为 [1,1,28,28]
        output = model(image.unsqueeze(0).to(device))
        _, predicted_label = torch.max(output, 1)  # 获取预测类别
        predicted_label = predicted_label.item()   # 转为 Python 整数

    # 显示图片
    ax.imshow(image.squeeze(), cmap='gray')        # 显示灰度图像

    # 根据预测是否正确设置标题颜色
    if predicted_label == true_label:
        color = 'green'   # 预测正确用绿色
    else:
        color = 'red'     # 预测错误用红色

    # 设置标题：显示真实标签和预测标签
    ax.set_title(f'真实: {true_label}  预测: {predicted_label}',
                 color=color, fontsize=12, fontweight='bold')
    ax.axis('off')        # 关闭坐标轴

plt.tight_layout()
plt.show()

## 第十三步：保存与加载模型

训练好的模型可以保存到本地文件，以便日后直接加载使用，无需重新训练：
- **保存**：使用 `torch.save()` 保存模型的 `state_dict`（参数字典）
- **加载**：使用 `torch.load()` + `model.load_state_dict()` 恢复模型参数
- 保存 `state_dict` 而非整个模型是 PyTorch 官方推荐的做法，更灵活、更安全

In [ ]:
# ===== 保存模型 =====
save_path = 'mnist_cnn.pth'                        # 定义保存路径
torch.save(model.state_dict(), save_path)           # 保存模型的参数字典
print(f"模型已保存到: {save_path}")

# ===== 加载模型 =====
# 创建一个新的模型实例（结构必须与保存时一致）
loaded_model = CNN().to(device)

# 加载之前保存的参数
# weights_only=True：只加载权重，更安全（避免反序列化攻击）
loaded_model.load_state_dict(torch.load(save_path, weights_only=True))
print("模型参数加载成功！")

# ===== 验证加载后的模型准确率 =====
loaded_model.eval()                                  # 切换到评估模式
verify_correct = 0                                   # 正确预测计数
verify_total = 0                                     # 总样本计数

with torch.no_grad():                                # 关闭梯度计算
    for images, labels in test_loader:
        images = images.to(device)                   # 图像移到设备
        labels = labels.to(device)                   # 标签移到设备
        outputs = loaded_model(images)               # 前向传播
        _, predicted = torch.max(outputs, 1)         # 取预测结果
        verify_total += labels.size(0)               # 累加总数
        verify_correct += (predicted == labels).sum().item()  # 累加正确数

verify_accuracy = 100.0 * verify_correct / verify_total
print(f"\n加载后的模型测试准确率: {verify_accuracy:.2f}%")
print(f"与保存前的准确率 ({test_accuracy:.2f}%) 一致: {abs(verify_accuracy - test_accuracy) < 0.01}")

---

## 总结

恭喜你完成了 MNIST 手写数字识别项目！回顾一下我们做了什么：

1. **导入库** → 准备 PyTorch 和可视化工具
2. **设置超参数** → 定义学习率、批量大小等
3. **加载数据** → 下载 MNIST 数据集并预处理
4. **数据探索** → 可视化图片和类别分布
5. **创建 DataLoader** → 按批次加载数据
6. **构建 CNN 模型** → 2 层卷积 + 2 层全连接
7. **定义损失和优化器** → 交叉熵 + Adam
8. **训练模型** → 10 个 epoch 的训练循环
9. **可视化训练过程** → 损失和准确率曲线
10. **测试评估** → 测试集准确率和各类别表现
11. **混淆矩阵** → 分析错误分类模式
12. **预测展示** → 直观查看模型效果
13. **保存/加载模型** → 持久化模型参数

**下一步可以尝试：**
- 增加数据增强（旋转、平移等）提高泛化能力
- 尝试更深的网络结构（如 ResNet）
- 使用学习率调度器动态调整学习率
- 在 Fashion-MNIST 等更具挑战性的数据集上实验